# 02 – Raw data check

Checks the 8 raw Berka files in the volume before the bronze load: file sizes, column names, row counts, empty values and key relationships.

## Raw file check

Compares the byte sizes of the 8 Berka files in the volume with their local sizes.

Output: for each file, the size in the volume, the expected size and a status (OK / DIFF / UNEXPECTED FILE). The total file count is printed at the end.

In [0]:
raw_path = "/Volumes/workspace/bronze/raw/"

# Byte sizes measured locally with `ls -la`
expected_sizes = {
    "account.csv": 155356,
    "card.csv": 31588,
    "client.csv": 94820,
    "disp.csv": 129716,
    "district.csv": 6574,
    "loan.csv": 27037,
    "order.csv": 273800,
    "trans.csv": 69406578,
}

file_list = dbutils.fs.ls(raw_path)

for file_info in file_list:
    file_name = file_info.name
    actual_size = file_info.size

    if file_name not in expected_sizes:
        status = "UNEXPECTED FILE"
        expected_size = None
    else:
        expected_size = expected_sizes[file_name]
        if actual_size == expected_size:
            status = "OK"
        else:
            status = "DIFF"

    print(f"{file_name:15} volume={actual_size:>10}  expected={expected_size}  {status}")

print(f"\nTotal files: {len(file_list)} (expected: {len(expected_sizes)})")

## Read with Spark and check column names

Reads the 8 files with `header=True` and `sep=";"` and checks the column names. `inferSchema` is not used, so all columns are strings and the raw values are shown as they are.

Output: column count and column names per table. Clean, unquoted column names mean the header, delimiter and quote settings are correct. The `district` table shows `A1`…`A16`.

In [0]:
raw_path = "/Volumes/workspace/bronze/raw/"
table_names = ["account", "card", "client", "disp", "district", "loan", "order", "trans"]

# Table name -> DataFrame; avoids reading the files again in the next cells
raw_dfs = {}

for table_name in table_names:
    file_path = raw_path + table_name + ".csv"
    df_raw = spark.read.option("header", True).option("sep", ";").csv(file_path)
    raw_dfs[table_name] = df_raw
    print(table_name, "->", len(df_raw.columns), "columns:", df_raw.columns)

## Row counts

Compares the row counts read by Spark with the expected values found with `wc -l` in Git Bash (excluding the header).

Output: Spark count, expected count and status per table. All should be `OK`, with a total of 1,079,680. `DIFF` means a row is split or merged during reading (quote or line ending issue).

In [0]:
expected_counts = {
    "account": 4500,
    "card": 892,
    "client": 5369,
    "disp": 5369,
    "district": 77,
    "loan": 682,
    "order": 6471,
    "trans": 1056320,
}

total_spark = 0

for table_name in table_names:
    spark_count = raw_dfs[table_name].count()
    expected_count = expected_counts[table_name]
    total_spark = total_spark + spark_count

    if spark_count == expected_count:
        status = "OK"
    else:
        status = "DIFF"

    print(table_name, "| spark:", spark_count, "| expected:", expected_count, "|", status)

print("Total:", total_spark, "| expected: 1079680")

## Empty values in trans

Shows whether `""` and `;;` in the raw file are read by Spark as NULL or as an empty string. Values with only whitespace are counted separately.

Output: null, empty and whitespace counts per column. These numbers are used to keep the raw values in bronze and to convert empty values to a single form (NULL) in silver.

In [0]:
from pyspark.sql import functions as F

df_trans = raw_dfs["trans"]

for column_name in df_trans.columns:
    col = F.col(column_name)
    null_count = df_trans.filter(col.isNull()).count()
    empty_count = df_trans.filter(col == "").count()
    whitespace_count = df_trans.filter((F.trim(col) == "") & (col != "")).count()
    print(column_name, "| null:", null_count, "| empty:", empty_count, "| whitespace:", whitespace_count)


## Key relationships between tables (orphan check)

Checks that every foreign key in a child table has a match in the parent table. A `left_anti` join returns only the child rows with no match in the parent.

Output: orphan count per relationship. All should be 0; a value above 0 would need separate handling in silver.

In [0]:
# (child table, child column, parent table, parent column)
relations = [
    ("client", "district_id", "district", "A1"),
    ("account", "district_id", "district", "A1"),
    ("disp", "client_id", "client", "client_id"),
    ("disp", "account_id", "account", "account_id"),
    ("card", "disp_id", "disp", "disp_id"),
    ("loan", "account_id", "account", "account_id"),
    ("order", "account_id", "account", "account_id"),
    ("trans", "account_id", "account", "account_id"),
]

for relation in relations:
    child_table = relation[0]
    child_col = relation[1]
    parent_table = relation[2]
    parent_col = relation[3]

    df_child = raw_dfs[child_table]
    df_parent = raw_dfs[parent_table]

    join_condition = df_child[child_col] == df_parent[parent_col]
    df_orphans = df_child.join(df_parent, join_condition, "left_anti")
    orphan_count = df_orphans.count()

    print(child_table + "." + child_col, "->", parent_table + "." + parent_col, "| orphans:", orphan_count)